# 01 — Camada de dados (DAL)

Desenvolve as funções de acesso a dados (baixar, gravar/ler SQLite, retornos). **F1, NF6.**

In [7]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import sqlite3
from contextlib import closing
from typing import Sequence
import pandas as pd
print('setup OK')

setup OK


In [24]:
print(os.getcwd())

print(os.listdir())

print(sys.version)

print(os.path.dirname(os.getcwd()))
print(os.path.dirname(_cwd))

c:\Users\MURILO\Desktop\TCC\Ambiente de Desenvolvimento\tests
['01_dados.ipynb', '02_calibracao.ipynb', '03_renda_fixa.ipynb', '04_renda_variavel.ipynb', '05_investidor.ipynb', '06_carteira_otima.ipynb', '07_momento_phi.ipynb', '08_recorrencia.ipynb', '09_consumo.ipynb', '10_propagacao.ipynb', '11_funcao_valor.ipynb', '12_miopia.ipynb', '13_consumo_crescente.ipynb', '14_convergencia_merton.ipynb', '15_pipeline.ipynb', '16_ingestao.ipynb', '17_entrypoint.ipynb', '18_validacao.ipynb', '19_algoritmo_quantecon.ipynb']
3.11.9 | packaged by Anaconda, Inc. | (main, Apr 19 2024, 16:40:41) [MSC v.1916 64 bit (AMD64)]
c:\Users\MURILO\Desktop\TCC\Ambiente de Desenvolvimento
c:\Users\MURILO\Desktop\TCC\Ambiente de Desenvolvimento


## Desenvolvimento

A(s) função(ões)/classe(s) abaixo foi(ram) escrita(s) aqui e, após os testes, movida(s) para `app/dal.py`.

In [1]:
def _validar_identificador(nome: str) -> None:
    """Impede nomes de tabela inválidos/injeção (o nome vem do código, mas
    validar mantém o contrato explícito)."""
    if not nome.isidentifier():
        raise ValueError(f"nome de tabela inválido: {nome!r}")

In [10]:
# teste

casos = [
    "clientes",
    "tabela1",
    "123tabela",
    "minha tabela",
    "clientes!",
    "cliente ",
    " ",
]

for nome in casos:
    try:
        _validar_identificador(nome)
        print(f"{nome!r} -> válido")
    except ValueError as e:                     # guarda ValueError em "e" e faz print do nome e da mensagem de erro
        print(f"{nome!r} -> inválido: ({e})")

'clientes' -> válido
'tabela1' -> válido
'123tabela' -> inválido: (nome de tabela inválido: '123tabela')
'minha tabela' -> inválido: (nome de tabela inválido: 'minha tabela')
'clientes!' -> inválido: (nome de tabela inválido: 'clientes!')
'cliente ' -> inválido: (nome de tabela inválido: 'cliente ')
' ' -> inválido: (nome de tabela inválido: ' ')


In [9]:
def baixar_precos(
    ativos: Sequence[str],
    inicio: str,
    fim: str | None = None,
    frequencia: str = "1mo",
) -> pd.DataFrame:
    """Baixa preços de fechamento (ajustado) do Yahoo Finance. (F1)

    Parameters
    ----------
    ativos : sequência de tickers, ex.: ``["^BVSP"]``.
    inicio : data inicial ``AAAA-MM-DD``.
    fim : data final ``AAAA-MM-DD`` (``None`` => hoje).
    frequencia : intervalo do yfinance (``"1mo"`` = mensal).

    Returns
    -------
    DataFrame com a coluna ``data`` (``AAAA-MM``) na primeira posição e uma
    coluna por ticker (preço de fechamento ajustado).

    Notes
    -----
    Usa a *chart API* pública do Yahoo via ``urllib`` (stdlib) — **sem**
    ``yfinance``/``curl_cffi``, que sofriam de erro de certificado SSL no
    Windows. Imports tardios mantêm os testes de banco/retorno offline (NF4).
    """
    import json
    from datetime import datetime, timezone
    from urllib.parse import quote
    from urllib.request import Request, urlopen

    def _epoch(d: str) -> int:
        return int(pd.Timestamp(d, tz="UTC").timestamp())

    p1 = _epoch(inicio)
    p2 = _epoch(fim or datetime.now(timezone.utc).strftime("%Y-%m-%d"))

    series: dict[str, pd.Series] = {}
    for tk in ativos:
        url = (f"https://query1.finance.yahoo.com/v8/finance/chart/{quote(tk)}"
               f"?period1={p1}&period2={p2}&interval={frequencia}") # prepara a URL para a API do Yahoo Finance, incluindo o ticker, período inicial e final, e a frequência de dados
        req = Request(url, headers={"User-Agent": "Mozilla/5.0"}) # prepara a requisição HTTP GET para o Yahoo Finance, incluindo um cabeçalho de agente de usuário para evitar bloqueios
        with urlopen(req, timeout=30) as resp: # envia a requisição HTTP GET para o Yahoo Finance e recebe a resposta
            payload = json.load(resp) # transforma a resposta JSON em um dicionário Python
        resultado = (payload.get("chart") or {}).get("result") # extrai a lista de resultados do payload JSON, ou uma lista vazia se não houver resultados
        if not resultado:
            raise ValueError(f"Yahoo não retornou dados para {tk!r}.")
        res = resultado[0] # pega o primeiro resultado da lista de resultados, que contém os dados do gráfico para o ticker solicitado
        ts = res.get("timestamp") or [] # extrai a lista de timestamps do resultado, ou uma lista vazia se não houver timestamps
        close = res["indicators"]["quote"][0].get("close") or [] # extrai a lista de preços de fechamento do resultado, ou uma lista vazia se não houver preços de fechamento
        meses = [datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m") for t in ts] # converte cada timestamp em uma string no formato "AAAA-MM" (ano-mês) usando o fuso horário UTC
        series[tk] = pd.Series(close, index=meses, name=tk) # cria uma série pandas com os preços de fechamento, usando os meses como índice e o ticker como nome da série

    df = pd.DataFrame(series).dropna(how="any") # cria um DataFrame pandas a partir das séries de preços de fechamento, descartando quaisquer linhas que contenham valores ausentes (NaN)
                                                # em qualquer coluna: dropna(how="any") remove linhas com NaN em qualquer coluna, garantindo que o DataFrame final contenha apenas meses com
                                                #  dados completos para todos os tickers.
    return df.reset_index().rename(columns={"index": "data"}) # reset_index() transforma o índice (meses) em uma coluna normal chamada "index", e rename(columns={"index": "data"}) 
                                                                # renomeia essa coluna para "data", resultando em um DataFrame final com a coluna "data" e uma coluna para cada ticker com os 
                                                                # preços de fechamento ajustados.

In [ ]:
# teste _epoch: converte data no formato AAAA-MM-DD para timestamp (segundos desde 1970-01-01 UTC)
# o "UTC" é necessário para garantir que a conversão seja feita no fuso horário correto, evitando discrepâncias devido a fusos horários locais. Por isso, UTC significa 
# "Coordinated Universal Time", que é o padrão de tempo usado para evitar confusões com fusos horários.

data = pd.Timestamp("2024-01-01", tz="UTC")
print(data)

2024-01-01 00:00:00+00:00


In [ ]:
# teste: testar url de download do Yahoo Finance para o ativo PETR4.SA, com período de 2020-01-01 a 2020-12-31 e frequência mensal.

series: dict[str, pd.Series] = {}
print(series)
print(type(series))

# pd.Series cria uma estrutura parecida com uma coluna de uma tbaela, com índice e valores. Aqui, estamos criando uma Series com os valores [10, 20, 30] 
# e atribuindo a chave "PETR4.SA" no dicionário series. Os índices são automaticamente gerados como 0, 1, 2. A Series resultante terá o nome "PETR4.SA".

series["PETR4.SA"] = pd.Series([10,20,30])
print(series)

series["VALE3.SA"] = pd.Series([100,200,300])
print(series)

series["ITUB4.SA"] = pd.Series([50,60,70], index=["2024-01", "2024-02", "2024-03"], name="ITUB4.SA")
print(series)

{}
<class 'dict'>
{'PETR4.SA': 0    10
1    20
2    30
dtype: int64}
{'PETR4.SA': 0    10
1    20
2    30
dtype: int64, 'VALE3.SA': 0    100
1    200
2    300
dtype: int64}
{'PETR4.SA': 0    10
1    20
2    30
dtype: int64, 'VALE3.SA': 0    100
1    200
2    300
dtype: int64, 'ITUB4.SA': 2024-01    50
2024-02    60
2024-03    70
Name: ITUB4.SA, dtype: int64}


In [ ]:
# teste: testar url de download do Yahoo Finance para o ativo PETR4.SA, com período de 2020-01-01 a 2020-12-31 e frequência mensal.

from urllib.parse import quote
import json

tk = "PETR4.SA"
p1 = 1577836800
p2 = 1609372800
frequencia = "1mo"

url = (
    f"https://query1.finance.yahoo.com/v8/finance/chart/{quote(tk)}"
    f"?period1={p1}&period2={p2}&interval={frequencia}"
)

print(url)


https://query1.finance.yahoo.com/v8/finance/chart/PETR4.SA?period1=1577836800&period2=1609372800&interval=1mo


In [26]:
# teste: testar url de download do Yahoo Finance para o ativo PETR4.SA, com período de 2020-01-01 a 2020-12-31 e frequência mensal.

from urllib.request import Request, urlopen

req = Request(url, headers={"User-Agent": "Mozilla/5.0"}) #criou o pedido HTTP com o cabeçalho User-Agent para simular um navegador, evitando bloqueios do Yahoo Finance: nada foi enviado 
                                                            # ainda, apenas criado o pedido HTTP.

print(type(req))

resp = urlopen(req, timeout=30) # enviou o pedido HTTP e recebeu a resposta do Yahoo Finance, que contém os dados solicitados.

try:
    payload = json.load(resp) # carregou o conteúdo da resposta HTTP (JSON) em um dicionário Python: lê o conteúdo da resposta HTTP e converte de JSON para um dicionário Python.
                                # o json.load() lê o conteúdo da resposta HTTP e converte de JSON para um dicionário Python. O resultado é armazenado na variável payload.
    print(payload)
finally:
    resp.close() # fechou a conexão HTTP para liberar recursos do sistema.

# Ou poderia direto o "with urlopen(req, timeout=30) as resp:", pois o "with" garante que a conexão seja fechada automaticamente ao final do bloco, mesmo que ocorra uma exceção.

with urlopen(req, timeout=30) as resp:
    payload = json.load(resp)
    print(payload)

print(payload.get("chart")) # acessa o dicionário "payload" e obtém o valor associado à chave "chart", que contém os dados do gráfico retornados pelo Yahoo Finance:
                            # get é usado para evitar KeyError caso a chave não exista, retornando None em vez disso. O resultado é um dicionário com as chaves "result" e "error".

"""
payload
│
└── "chart"
      │
      ├── "result"
      │      │
      │      └── [...]
      │
      └── "error"
"""

pessoa = {
    "nome": "Carlos",
    "idade": 25
}

print(pessoa["nome"])
print(pessoa.get("nome"))
# print(pessoa["altura"]) # gera KeyError, pois a chave "altura" não existe no dicionário.
print(pessoa.get("altura")) # retorna None, pois a chave "altura" não existe no dicionário. O método get() é útil para evitar KeyError ao acessar chaves que podem não estar presentes no dicionário.


print((payload.get("chart") or {}).get("result")) # acessa o dicionário "payload", obtém o valor associado à chave "chart" (ou um dicionário vazio se não existir) e, em seguida, obtém o valor 
                                                    # associado à chave "result" (ou None se não existir). O resultado é uma lista de resultados do gráfico retornados pelo Yahoo Finance.

resultado = (payload.get("chart") or {}).get("result")

"""
payload
│
└── chart
      │
      └── result
            │
            └── lista de dados
"""

if not resultado: # verifica se a variável "resultado" está vazia ou é None. Se estiver vazia, significa que o Yahoo Finance não retornou dados para o ticker solicitado.
                    # not verifica se a expressão é falsa, e ela é falsa quando a variável é None, False, 0, "", [], {}, etc. Aqui, se resultado for None ou uma lista vazia, 
                    # a condição será verdadeira.
    raise ValueError(f"Yahoo não retornou dados para {tk!r}.") # levanta um ValueError com uma mensagem indicando que o Yahoo Finance não retornou dados para o ticker especificado.

valores = [
    None, 
    [],
    {},
    "",
    [1],
    {"a":1},
    "Python"
]

for v in valores:
    print(v, "->", bool(v)) # bool() converte o valor para um booleano, retornando True para valores "verdadeiros" e False para valores "falsos". Aqui, estamos testando diferentes 
                                # tipos de valores para ver como eles são avaliados em termos de verdade/falsidade.

for v in valores:
    if not v:
        print(v, "é considerado vazio pelo Python")

res = resultado[0] # pega o primeiro resultado da lista de resultados, que contém os dados do gráfico para o ticker solicitado.
print(res)

ts = res.get("timestamp") or [] # procura a chave "timestamp" no dicionário "res". Se a chave existir, retorna o valor associado (uma lista de timestamps); se não existir, retorna uma lista vazia.
print(ts)

close = res["indicators"]["quote"][0].get("close") or [] # acessa o dicionário "res", obtém a lista de indicadores, acessa o primeiro indicador (que contém os preços) e, em seguida, obtém a lista de 
                                                            # preços de fechamento.

print(res["indicators"])
print(res["indicators"]["quote"][0])
print(close)

<class 'urllib.request.Request'>
{'chart': {'result': [{'meta': {'currency': 'BRL', 'symbol': 'PETR4.SA', 'exchangeName': 'SAO', 'fullExchangeName': 'São Paulo', 'instrumentType': 'EQUITY', 'firstTradeDate': 946900800, 'regularMarketTime': 1783347387, 'hasPrePostMarketData': False, 'gmtoffset': -10800, 'timezone': 'BRT', 'exchangeTimezoneName': 'America/Sao_Paulo', 'regularMarketPrice': 37.83, 'fiftyTwoWeekHigh': 50.69, 'fiftyTwoWeekLow': 29.31, 'regularMarketDayHigh': 38.02, 'regularMarketDayLow': 37.61, 'regularMarketVolume': 6447400, 'longName': 'Petróleo Brasileiro S.A. - Petrobras', 'shortName': 'PETROBRAS   PN      N2', 'chartPreviousClose': 30.18, 'priceHint': 2, 'currentTradingPeriod': {'pre': {'timezone': 'BRT', 'start': 1783341900, 'end': 1783342800, 'gmtoffset': -10800}, 'regular': {'timezone': 'BRT', 'start': 1783342800, 'end': 1783368000, 'gmtoffset': -10800}, 'post': {'timezone': 'BRT', 'start': 1783368000, 'end': 1783371600, 'gmtoffset': -10800}}, 'dataGranularity': '1mo

In [5]:
# teste

from datetime import datetime, timezone

ts = [
    1704067200,
    1706745600,
    1709251200
]

for t in ts:
    print("Epoch:", t)
    print("Data :", datetime.fromtimestamp(t, tz=timezone.utc)) # converte o timestamp (epoch) para uma data legível no fuso horário UTC.
    print("Texto:", datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m")) # formata a data como uma string no formato "AAAA-MM" (ano-mês) usando o fuso horário UTC.

print(ts)

Epoch: 1704067200
Data : 2024-01-01 00:00:00+00:00
Texto: 2024-01
Epoch: 1706745600
Data : 2024-02-01 00:00:00+00:00
Texto: 2024-02
Epoch: 1709251200
Data : 2024-03-01 00:00:00+00:00
Texto: 2024-03
[1704067200, 1706745600, 1709251200]


In [12]:
# teste

df = baixar_precos(
    ativos=["PETR4.SA"],
    inicio="2020-01-01",
)

print(df)

       data   PETR4.SA
0   2020-01  28.450001
1   2020-02  25.340000
2   2020-03  13.990000
3   2020-04  18.049999
4   2020-05  20.340000
..      ...        ...
74  2026-03  48.669998
75  2026-04  49.080002
76  2026-05  42.000000
77  2026-06  37.799999
78  2026-07  38.250000

[79 rows x 2 columns]


In [ ]:
def baixar_cdi_bcb(inicio: str, fim: str | None = None) -> pd.DataFrame:
    """Baixa o CDI mensal da API SGS do Banco Central. (F1)

    Série 4391 = 'Taxa de juros - CDI acumulada no mês' (% a.m.). Devolve um
    DataFrame com ``data`` (AAAA-MM) e ``cdi`` (decimal por mês, ex.: 0.0034).
    Fonte oficial, gratuita e sem cadastro; requer internet.

    Notes
    -----
    Imports tardios (``json``/``urllib``, stdlib) e nenhuma dependência nova.
    """
    import json
    from datetime import date as _date
    from urllib.request import urlopen

    di = pd.to_datetime(inicio).strftime("%d/%m/%Y")
    dfim = pd.to_datetime(fim or _date.today().isoformat()).strftime("%d/%m/%Y")
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados"
           f"?formato=json&dataInicial={di}&dataFinal={dfim}")
    with urlopen(url, timeout=30) as resp:
        dados = json.load(resp) #json.load() lê o conteúdo da resposta HTTP (JSON) e converte para um objeto Python (lista de dicionários). O resultado é armazenado na variável "dados".
    if not dados:
        raise ValueError("BCB não retornou CDI para o período pedido.")
    df = pd.DataFrame(dados)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y").dt.strftime("%Y-%m") # converte toda coluna data do DataFrame para o formato AAAA-MM, usando o método pd.to_datetime() 
                                                                                    # para interpretar as datas no formato DD/MM/AAAA e, em seguida, aplicando .dt.strftime("%Y-%m") para 
                                                                                    # formatar as datas como strings no formato desejado.
    df["cdi"] = df["valor"].astype(float) / 100.0   # % a.m. -> decimal # transforma a coluna "valor" (que contém o CDI em porcentagem) em decimal, dividindo por 100.0. O resultado é armazenado na nova coluna "cdi".
    return df[["data", "cdi"]]

In [17]:
# teste: transforma timestamp em data legível e formata como string no formato "AAAA-MM".

inicio = "2024-01-01"

di = pd.to_datetime(inicio)

print(di)
print(type(di))

print(di.strftime("%d/%m/%Y"))
print(type(di.strftime("%d/%m/%Y")))

2024-01-01 00:00:00
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
01/01/2024
<class 'str'>


In [21]:
# teste

df = baixar_cdi_bcb(inicio="2020-01-01")

print(df)

       data     cdi
0   2020-01  0.0038
1   2020-02  0.0029
2   2020-03  0.0034
3   2020-04  0.0028
4   2020-05  0.0024
..      ...     ...
74  2026-03  0.0121
75  2026-04  0.0109
76  2026-05  0.0107
77  2026-06  0.0112
78  2026-07  0.0016

[79 rows x 2 columns]


In [11]:
def calcular_retornos(precos: pd.DataFrame, coluna_data: str = "data") -> pd.DataFrame:
    """Converte preços em retornos simples mensais. (F1)

    A primeira observação é descartada (não há retorno anterior), portanto
    ``T_efetivo = T − 1``. Todas as colunas que não sejam ``coluna_data`` são
    tratadas como séries de preço.

    Notes
    -----
    A taxa CDI, por **já ser** um retorno (não um preço), não passa por aqui:
    ela entra direto na coluna ``cdi`` da tabela ``retornos``.
    """
    if coluna_data not in precos.columns:
        raise KeyError(f"coluna '{coluna_data}' ausente em precos.")

    datas = precos[coluna_data].iloc[1:].to_numpy()
    numericas = precos.drop(columns=[coluna_data])
    ret = numericas.pct_change(fill_method=None).iloc[1:].reset_index(drop=True)
    ret.insert(0, coluna_data, datas)
    return ret

In [43]:
# teste
 
precos = pd.DataFrame({
    "data": ["2024-01", "2024-02", "2024-03"],
    "PETR4": [100, 110, 105],
    "VALE3": [50, 55, 60]
})

print(precos)

print(precos.columns)

for coluna in precos.columns:
    print(coluna)

print("data" in precos.columns)

precos = pd.DataFrame({
    "data": ["2024-01", "2024-02"],
    "PETR4": [100, 110]
})

print(precos)
print(precos.columns)

print("data" in precos.columns)
print("PETR4" in precos.columns)
print("VALE3" in precos.columns)
print("VALE3" not in precos.columns)

      data  PETR4  VALE3
0  2024-01    100     50
1  2024-02    110     55
2  2024-03    105     60
Index(['data', 'PETR4', 'VALE3'], dtype='object')
data
PETR4
VALE3
True
      data  PETR4
0  2024-01    100
1  2024-02    110
Index(['data', 'PETR4'], dtype='object')
True
True
False
True


In [ ]:
# teste

print(precos["data"])

print(precos["data"].iloc[1:]) # .iloc[1:] seleciona todas as linhas a partir da segunda (índice 1) da coluna "data", retornando uma Series com os valores ["2024-02"].
print(type(precos["data"].iloc[1:])) # <class 'pandas.core.series.Series'>

datas = precos["data"].iloc[1:].to_numpy() # .to_numpy() converte a Series em um array NumPy, que é uma estrutura de dados eficiente para operações numéricas. O resultado é armazenado na variável "datas".
print(datas)
print(type(datas))

numericas = precos.drop(columns=["data"]) # .drop(columns=["data"]) cria um novo DataFrame sem a coluna "data", mantendo apenas as colunas numéricas (preços). O resultado é armazenado na variável "numericas".

ret = numericas.pct_change(fill_method=None) # fill_method=None evita que o pandas preencha valores ausentes (NaN) durante o cálculo da variação percentual. O resultado é um DataFrame com os retornos percentuais, incluindo NaN na primeira linha, pois não há retorno anterior para calcular.

print(ret) # .pct_change() calcula a variação percentual entre cada linha e a linha anterior, ou seja, preço atual menos preço anterior dividido pelo preço anterior. 
            # O resultado é um DataFrame com os retornos percentuais.

ret = numericas.pct_change().iloc[1:]
print(ret) # .iloc[1:] seleciona todas as linhas a partir da segunda (índice 1) do DataFrame de retornos, descartando a primeira linha que contém NaN (não há retorno anterior para calcular).

ret = numericas.pct_change().iloc[1:].reset_index(drop=True) # .reset_index(drop=True) redefine o índice do DataFrame de retornos, descartando o índice antigo (drop=True). O resultado é um 
                                                                # DataFrame com índices sequenciais começando em 0.
print(ret)

ret.insert(0, "data", datas) # .insert(0, "data", datas) insere a coluna "data" no DataFrame de retornos na posição 0 (primeira coluna), usando os valores do array "datas". O resultado é um 
                                # DataFrame com a coluna "data" seguida pelas colunas de retornos. 0 é a posição onde a nova coluna será inserida, "data" é o nome da nova coluna e "datas" é 
                                # o array de valores que será usado para preencher a coluna.
print(ret)

0    2024-01
1    2024-02
Name: data, dtype: object
1    2024-02
Name: data, dtype: object
<class 'pandas.core.series.Series'>
['2024-02']
<class 'numpy.ndarray'>
   PETR4
0    NaN
1    0.1
   PETR4
1    0.1
   PETR4
0    0.1
      data  PETR4
0  2024-02    0.1


In [46]:
# teste

precos = pd.DataFrame({
    "data": ["2024-01", "2024-02", "2024-03", "2024-04"],
    "PETR4": [100, 110, 105, 120],
    "VALE3": [50, 55, 60, 58]
})

print(precos)

retornos = calcular_retornos(precos, coluna_data="data")
print(retornos)

      data  PETR4  VALE3
0  2024-01    100     50
1  2024-02    110     55
2  2024-03    105     60
3  2024-04    120     58
      data     PETR4     VALE3
0  2024-02  0.100000  0.100000
1  2024-03 -0.045455  0.090909
2  2024-04  0.142857 -0.033333


In [12]:
def gravar_sqlite(
    df: pd.DataFrame,
    db_path: str,
    tabela: str,
    if_exists: str = "replace",
) -> None:
    """Grava um DataFrame numa tabela do banco SQLite. (F1, NF6)

    Usa ``contextlib.closing`` porque ``with sqlite3.connect(...)`` apenas
    gerencia a transação — **não fecha** a conexão; sem fechar, o arquivo do
    banco fica travado no Windows (NF3).
    """
    _validar_identificador(tabela)
    with closing(sqlite3.connect(db_path)) as con: #sqlite3.connect(db_path) cria uma conexão com o banco de dados SQLite especificado pelo caminho db_path. A conexão é usada para 
                                                    # executar comandos SQL e interagir com o banco de dados. closing() garante que a conexão seja fechada automaticamente ao final do bloco, 
                                                    # mesmo que ocorra uma exceção. Isto serve para evitar que o arquivo do banco de dados fique travado no Windows, pois o SQLite mantém o 
                                                    # arquivo aberto enquanto a conexão estiver ativa. con é a variável que representa a conexão com o banco de dados SQLite, e é usada para 
                                                    # executar comandos SQL e interagir com o banco de dados.
        df.to_sql(tabela, con, if_exists=if_exists, index=False) # to_sql() grava o DataFrame df em uma tabela do banco de dados SQLite representado pela conexão con. O parâmetro tabela 
                                                                    # especifica o nome da tabela, if_exists define o comportamento caso a tabela já exista (substituir, adicionar ou falhar), 
                                                                    # e index=False indica que o índice do DataFrame não deve ser gravado como uma coluna na tabela.
        con.commit() # .commit() confirma a transação atual, garantindo que todas as alterações feitas no banco de dados (como a gravação do DataFrame) sejam salvas permanentemente.

In [15]:
# teste

df = pd.DataFrame({
    "nome": ["João", "Maria"],
    "idade": [20, 25]
})

print(df)

db_path = "C:\\Users\\MURILO\\Desktop\\TCC\\Ambiente de Desenvolvimento\\teste.db" # caminho do arquivo do banco de dados SQLite. Pode ser um caminho absoluto ou relativo. Se o arquivo não existir, ele será criado automaticamente.

tabela = "clientes" # é o nome da tabela que será criada no banco de dados SQLite. O nome deve ser um identificador válido, sem espaços ou caracteres especiais.

gravar_sqlite(df, db_path, tabela, if_exists="replace") # chama a função gravar_sqlite() para gravar o DataFrame df na tabela clientes do banco de dados SQLite especificado pelo caminho db_path.


    nome  idade
0   João     20
1  Maria     25


In [17]:
def ler_sqlite(db_path: str, tabela: str) -> pd.DataFrame:
    """Lê uma tabela do banco SQLite para um DataFrame. (F1, NF6)"""
    _validar_identificador(tabela)
    with closing(sqlite3.connect(db_path)) as con:
        return pd.read_sql(f"SELECT * FROM {tabela}", con)


In [18]:
# teste

ler_sqlite(db_path, tabela)

,nome,idade
0,João,20
1,Maria,25


**Teste** — retornos, round-trip SQLite e downloads reais (protegidos).

In [ ]:
import pandas as pd, os, tempfile
precos = pd.DataFrame({'data': ['2000-01','2000-02','2000-03','2000-04'], 
                       'ibov': [100.,102.,99.,105.]})
print('calcular_retornos:'); print(calcular_retornos(precos))

esp = precos['ibov'].pct_change(fill_method=None).iloc[1:].reset_index(drop=True)
assert calcular_retornos(precos)['ibov'].round(10).tolist() == esp.round(10).tolist()
db = os.path.join(tempfile.gettempdir(), 'dev01.db'); gravar_sqlite(precos, db, 'ibovespa')
lido = ler_sqlite(db, 'ibovespa'); pd.testing.assert_frame_equal(precos.reset_index(drop=True), lido, check_dtype=False); os.remove(db)
print('round-trip SQLite: OK')

try:
    print(baixar_precos(['^BVSP'], '2023-01-01', '2023-04-01'))
    print(baixar_cdi_bcb('2023-01-01', '2023-04-01'))
except Exception as e:
    print('download offline:', type(e).__name__, e)
print('OK')

calcular_retornos:
      data      ibov
0  2000-02  0.020000
1  2000-03 -0.029412
2  2000-04  0.060606
round-trip SQLite: OK
      data   ibov
0  2000-01  100.0
1  2000-02  102.0
2  2000-03   99.0
3  2000-04  105.0
      data     ^BVSP
0  2023-01  113532.0
1  2023-02  104932.0
2  2023-03  101882.0
download offline: JSONDecodeError Expecting value: line 1 column 1 (char 0)
OK


In [31]:
# teste

print(tempfile.gettempdir()) # gettempdir() retorna o caminho do diretório temporário do sistema operacional, que é usado para armazenar arquivos temporários. No Windows, geralmente 
                                # é algo como "C:\Users\<username>\AppData\Local\Temp", enquanto no Linux e macOS é "/tmp". Este diretório é útil para criar arquivos temporários que não
                                # precisam ser mantidos após a execução do programa: "get temp dir" significa "obter diretório temporário". O resultado é uma string com o caminho do diretório temporário.

pasta = tempfile.gettempdir()

print(type(pasta)) # <class 'str'>, pois gettempdir() retorna uma string com o caminho do diretório temporário.

print(os.path.join(pasta, 'dev01.db')) # join() combina o caminho do diretório temporário com o nome do arquivo 'dev01.db', criando um caminho completo para o arquivo de banco de dados SQLite 
                                        # temporário. O join significa juntar, e ele junta partes de um caminho: primeira parte é a pasta temporário, segunda parte é o nome do arquivo. O resultado 
                                        # é uma string com o caminho completo para o arquivo de banco de dados temporário, como, por exemplo, "C:\Users\<username>\AppData\Local\Temp\dev01.db" no 
                                        # Windows ou "/tmp/dev01.db" no Linux/macOS.

print(os.path.join("A", "B", "C"))

C:\Users\MURILO\AppData\Local\Temp
<class 'str'>
C:\Users\MURILO\AppData\Local\Temp\dev01.db
A\B\C


In [ ]:
# teste

df1 = pd.DataFrame({
    "nome": ["Ana", "João"],
    "idade": [20, 25]
})

df2 = pd.DataFrame({
    "nome": ["Ana", "João"],
    "idade": [20, 25]
})

pd.testing.assert_frame_equal(df1, df2, check_dtype=False) # assert_frame_equal() compara dois DataFrames e levanta um AssertionError se eles não forem iguais. O parâmetro check_dtype=False 
                                                            # indica que a comparação deve ignorar diferenças de tipo de dados entre as colunas. Se os DataFrames forem iguais, o código 
                                                            # continua sem erros; caso contrário, um erro será levantado.

df3 = pd.DataFrame({
    "A":[1,2]
})

df4 = pd.DataFrame({
    "A":[1.0,2.0]
})

print(df3.dtypes)
print(df4.dtypes)

pd.testing.assert_frame_equal(df3, df4, check_dtype=False) # check_dtype=False permite quseja, vai 
                                                            # ignorar se uma coluna é do tipo int64 e a outra é do tipo float64, desde que os valores sejam equivalentes.

A    int64
dtype: object
A    float64
dtype: object


✔ **Testado e aprovado — código movido para `app/dal.py`.**

Funções da DAL desenvolvidas e testadas.